In [1]:
# import sys, time
# from pprint import pprint
# # time.sleep(10)

# pprint(sys.version)
# from pathlib import Path

# HOME_DIR = Path.home()
# CODE_DIR = HOME_DIR / 'synthesizrr' / 'src'
# sys.path.insert(-2, '/home/ec2-user/anaconda3/envs/hft4/lib/python3.11/site-packages')
# sys.path.insert(-2, '/home/ec2-user/anaconda3/envs/hft4/bin/')
# sys.path.insert(-2, str(CODE_DIR))
# pprint(sys.path)

# import synthergent
# from typing import *
# import time, glob, os, sys, boto3, numpy as np, pandas as pd, json, requests, gc
# from pandas.core.frame import DataFrame as PandasDataFrame, Series as PandasSeries
# from pathlib import Path

# RAY_TMP_DIR = '/tmp/ray/'

# from synthergent.base.util import *
# from synthergent.base.data import *
# from synthergent.base.constants import *
# from synthergent.base.framework import *
# from synthergent.base.data.reader import Reader
# from synthergent.base.data.writer import Writer
# from synthergent.base.framework.dl.torch import *
# from synthergent.base.framework.task_data import DataSplit, Datasets, TaskData
# import synthergent.base.algorithm
# import synthergent.base.metric
# from synthergent.base.framework.task.classification import _normalize_label
# import datasets as ds
# # from datasets import load_dataset, load_from_disk
# from synthergent.base.framework.trainer.RayTuneTrainer import _ray_metric_str
# from termcolor import COLORS as TERMCOLOR_COLORS
# from termcolor import colored
# import ray
# from ray.util.dask import ray_dask_get, enable_dask_on_ray, disable_dask_on_ray

# from pprint import pprint

# os.environ['CUDA_VISIBLE_DEVICES'] = '4,6'

# ## print = Tracker.default().info

# # import hvplot.pandas
# # import holoviews as hv
# # import plotly.express as px
# # import plotly.io as pio
# # from IPython.display import display
# # from bokeh.palettes import Spectral, Set2, Set3

# # pio.templates.default = 'plotly_white'
# # hvplot.extension('plotly')
# # import numpy as np
# # import pandas as pd
# # import plotly.graph_objects as go
# # import plotly.io as pio
# # # pio.renderers.default='iframe'
# # # hvplot.extension('bokeh')

# # from bokeh.resources import INLINE as BOKEH_INLINE
# # from bokeh.io import output_notebook as bokeh_output_notebook
# # bokeh_output_notebook(BOKEH_INLINE)


# from synthergent import Synthergent, Cleaner
# import synthergent.cleaner

'3.11.8 | packaged by conda-forge | (main, Feb 16 2024, 20:53:32) [GCC 12.3.0]'
['/opt/conda/envs/hft4/lib/python311.zip',
 '/opt/conda/envs/hft4/lib/python3.11',
 '/opt/conda/envs/hft4/lib/python3.11/lib-dynload',
 '/home/ec2-user/anaconda3/envs/hft4/lib/python3.11/site-packages',
 '/home/ec2-user/anaconda3/envs/hft4/bin/',
 '/efs/litmus-server/users/adivekar/synthesizrr/src',
 '',
 '/opt/conda/envs/hft4/lib/python3.11/site-packages']


In [1]:
import sys
from pathlib import Path
BASE_DIR = Path.cwd()
SYNTHERGENT_DIR = BASE_DIR / 'src' 
sys.path.insert(-2, str(SYNTHERGENT_DIR))
from synthergent.util import *
from synthergent.constants import *

WARNING 03-29 13:26:47 _custom_ops.py:20] Failed to import from vllm._C with ModuleNotFoundError("No module named 'vllm._C'")
INFO 03-29 13:26:47 importing.py:15] Triton not installed or not compatible; certain GPU-related functions will not be available.


In [2]:
common_crawl_index_fm = FileMetadata.of(
    path='s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/',
    format='parquet',
)
common_crawl_index_files: List[str] = common_crawl_index_fm.list(file_glob='*.parquet')
for i, x in enumerate(common_crawl_index_files):
    print(f'[{i:03}] {x}')

ClientError: An error occurred (ExpiredToken) when calling the ListObjects operation: The provided token has expired.

# Task-oriented Cleaners: Extracting coffee articles from CommonCrawl

In [3]:
import synthergent.cleaner

coffee_articles = Synthergent.of(
    Cleaner.of(
        'StringCleaner',
        params=dict(
            col='url',
            cleaner=lambda url: urllib.parse.unquote(str(url)),
        )
    ),
    Cleaner.of(
        'StringCleaner',
        params=dict(
            col='url',
            cleaner=lambda url: str(url).replace('-', ' '),
        )
    ),
    Cleaner.of(
        'FilterColumnsByValue', 
        params=dict(
            contains={
                'url': [
                    'coffee', 
                    'espresso', 'french press', 'pour over', 'aeropress', 'chemex', 'cold brew',
                    'americano', 'latte', 'cappuccino', 'macchiato', 'flat white', 'mocha', 'affogato',
                    'arabica', 'robusta', 'liberica', 'excelsa', 'typica', 
                    'light roast', 'medium roast', 'dark roast', 'roast profile',
                ],
            }
        )
    )
)

# Extracting coffee articles from 1 million rows

In [8]:
common_crawl_index_sample = pd.read_parquet(random.Random(42).choice(common_crawl_index_files))[
    ['url', 'content_languages']
].sample(n=int(1e6), random_state=123)
with pd_display() as display:
    display(common_crawl_index_sample.sample(n=10, random_state=123))

,url,content_languages
4343529,https://twitcasting.tv/c:laguna_shimokita/shopcart/252486,eng
5257984,https://clairelife.tw/album/photo/141773808,zho
677224,https://www.bigo.tv/402319749,"eng,zho,ron"
3810139,https://skiweltcup.tv/index.php/oesv-news-oesterreichische-speeddamen-in-bansko-abgeschlagen/,"deu,eng"
1997297,https://tss.ib.tv/boxing/tag/jose-zaragoza,eng
7043786,https://www.happyhair.com.tw/designer/061-mio/,"zho,eng"
3622948,https://www.salve.tv/web/de/werbung/werbeclicks.php?click_db=&werbung_ID=65&videopool_ID=21303&URL=https://www.salve.tv/tv/topmeldungen/?rubrik=370,None
6918697,https://gogodesign.com.tw/process2.php,"zho,eng"
7149449,https://www.hongo.com.tw/pages/%E5%90%B9%E9%A2%A8%E6%A9%9F%E8%AD%B7%E9%AB%AE%E7%94%A8%E9%81%8E%E5%B0%B1%E5%85%A5%E5%9D%91,"zho,eng"
1903289,https://hgoah.tv/rubrique/h-play/culture/,"fra,eng"


In [9]:
print(f'{len(common_crawl_index_sample) / 1e6:.1f} million rows')

1.0 million rows


## Single CPU

In [10]:
cc_coffee_articles_sample = coffee_articles(data=common_crawl_index_sample)
cc_coffee_articles_sample

Synthergent:   0%|          | 0/3 [00:00<?, ?step/s]

,url,content_languages
0,https://www.almondcoffee.com.tw/not_member_ord...,"zho,eng"
1,https://aircoffee.com.tw/pageDetailed.php?page...,zho
2,https://www.coffeecenter.com.tw/Product_Catalo...,zho
3,https://watch.weareo.tv/videos/arabica trailer 1,eng
4,https://www.e classical.com.tw/coffee_aod_list...,"zho,eng"
...,...,...
1086,https://cheng10coffee.com.tw/tw/?utm_source=do...,"zho,eng,jpn"
1087,https://blog.academia.tv/tag/coffeetails/,"ita,eng"
1088,http://www.coffee tea.tv/2017/11/,"zho,eng"
1089,https://tvoutlet.tv/product categorie/keuken/a...,"nld,eng"


## 5 CPUs

In [11]:
cc_coffee_articles_sample = coffee_articles(
    data=common_crawl_index_sample,
    scaling=dict(
        parallelize='processes',
        max_workers=5,
        batch_size=200e3,
    )
)
cc_coffee_articles_sample

I0000 00:00:1734326435.038266   35622 work_stealing_thread_pool.cc:320] WorkStealingThreadPoolImpl::PrepareFork
I0000 00:00:1734326435.038481   35622 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1734326435.158612   35622 work_stealing_thread_pool.cc:320] WorkStealingThreadPoolImpl::PrepareFork
I0000 00:00:1734326435.158836   35622 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1734326435.265446   35622 work_stealing_thread_pool.cc:320] WorkStealingThreadPoolImpl::PrepareFork
I0000 00:00:1734326435.265660   35622 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1734326435.370082   35622 work_stealing_thread_pool.cc:320] WorkStealingThreadPoolImpl::PrepareFork
I0000 00:00:1734326435.370323   35622 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1734326435.478749   35622 wo

Synthergent:   0%|          | 0/3 [00:00<?, ?step/s]

,url,content_languages
0,https://www.almondcoffee.com.tw/not_member_ord...,"zho,eng"
1,https://aircoffee.com.tw/pageDetailed.php?page...,zho
2,https://www.coffeecenter.com.tw/Product_Catalo...,zho
3,https://watch.weareo.tv/videos/arabica trailer 1,eng
4,https://www.e classical.com.tw/coffee_aod_list...,"zho,eng"
...,...,...
1086,https://cheng10coffee.com.tw/tw/?utm_source=do...,"zho,eng,jpn"
1087,https://blog.academia.tv/tag/coffeetails/,"ita,eng"
1088,http://www.coffee tea.tv/2017/11/,"zho,eng"
1089,https://tvoutlet.tv/product categorie/keuken/a...,"nld,eng"


# Extracting coffee articles from 100 million rows

In [4]:
common_crawl_index_fm_smaller = FileMetadata.of(
    path='s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/',
    format='parquet',
    file_glob='part-0000*.parquet',
)
len(common_crawl_index_fm_smaller.list())

10

In [5]:
import ray, dask
from ray.util.dask import enable_dask_on_ray
ray.shutdown()
pprint(ray.init(
    address='ray://10.0.152.54:10001',
    ignore_reinit_error=True,
    _temp_dir=str(RAY_TMP_DIR),
    runtime_env={"py_modules": [
        synthergent,
    ]},
))
enable_dask_on_ray()
pprint(ray.cluster_resources())

2024-12-16 05:16:02,169	INFO client_builder.py:243 -- Passing the following kwargs to ray.init() on the server: ignore_reinit_error
I0000 00:00:1734326162.186159   35622 config.cc:230] gRPC experiments enabled: call_status_override_on_cancellation, event_engine_dns, event_engine_listener, http2_stats_fix, monitoring_experiment, pick_first_new, trace_record_callops, work_serializer_clears_time_cache
2024-12-16 05:16:02,710	INFO packaging.py:530 -- Creating a file package for local directory '/efs/litmus-server/users/adivekar/synthesizrr/src/synthergent'.
2024-12-16 05:16:03,057	INFO packaging.py:358 -- Pushing file package 'gcs://_ray_pkg_ec50077cedbf2a4c.zip' (4.31MiB) to Ray cluster...
2024-12-16 05:16:03,109	INFO packaging.py:371 -- Successfully pushed file package 'gcs://_ray_pkg_ec50077cedbf2a4c.zip'.
SIGTERM handler is not set because current thread is not the main thread.


ClientContext(dashboard_url='127.0.0.1:8265',
              python_version='3.11.8',
              ray_version='2.9.2',
              ray_commit='fce7a361807580953364e2da964f9498f3123bf9',
              protocol_version='2023-06-27',
              _num_clients=1,
              _context_to_restore=<ray.util.client._ClientContext object at 0x7f4d7d713310>)
{'CPU': 288.0,
 'memory': 1343201246208.0,
 'node:10.0.145.110': 1.0,
 'node:10.0.145.189': 1.0,
 'node:10.0.152.54': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 1020054732800.0}


/opt/conda/envs/hft4/lib/python3.11/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(
(dask:('total_mem_usage-1ba2a2d03f1db10ca0658c91adfe738f', 133) pid=77607, ip=10.0.145.110) /opt/conda/envs/hft4/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
(dask:('total_mem_usage-1ba2a2d03f1db10ca0658c91adfe738f', 133) pid=77607, ip=10.0.145.110)   _torch_pytree._register_pytree_node(
(dask:('total_mem_usage-1ba2a2d03f1db10ca0658c91adfe738f', 135) pid=77606, ip=10.0.145.110) /opt/conda/envs/hft4/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
(dask:('total_mem_usage-1ba2a2d03f1db10ca0658c91

In [6]:
with Timer():
    common_crawl_index = Reader.of(
        'parquet',
        data_schema={
            'url': 'text', 
            'content_languages': 'text',
        },
    ).read(
        common_crawl_index_fm_smaller,
        read_as='dask',
    )

Started at 2024-12-16T05:16:19.937473+00:00...
...completed in 3.64 seconds.


/opt/conda/envs/hft4/lib/python3.11/site-packages/ray/util/client/worker.py:614: UserWarning: More than 10MB of messages have been created to schedule tasks on the server. This can be slow on Ray Client due to communication overhead over the network. If you're running many fine-grained tasks, consider running them inside a single remote function. See the section on "Too fine-grained tasks" in the Ray Design Patterns document for more details: https://docs.google.com/document/d/167rnnDFIVRhHhK4mznEIemOtj63IOhtIPvSYaPgI4Fg/edit#heading=h.f7ins22n6nyl. If your functions frequently use large objects, consider storing the objects remotely with ray.put. An example of this is shown in the "Closure capture of large / unserializable object" section of the Ray Design Patterns document, available here: https://docs.google.com/document/d/167rnnDFIVRhHhK4mznEIemOtj63IOhtIPvSYaPgI4Fg/edit#heading=h.1afmymq455wu
  warnings.warn(


In [7]:
print(f'{len(common_crawl_index) / 1e6:.1f} million rows')

105.8 million rows


In [7]:
cc_coffee_articles = coffee_articles(
    data=common_crawl_index,
    scaling=dict(
        parallelize='ray',
        partition_size='100MB',
    )
)
cc_coffee_articles

Synthergent:   0%|          | 0/3 [00:00<?, ?step/s]

,content_languages,url
1861,"por,eng",http://blogdocurioso1.blogspot.com/2011/09/ame...
2172,por,https://blogdoeduardopeixoto.blogspot.com/2010...
2690,por,https://blogdoeduardopeixoto.blogspot.com/2012...
3620,por,https://blogdoespacoaberto.blogspot.com/2009/0...
10291,por,https://blogdolaert.blogspot.com/2023/01/deput...
...,...,...
1181949,eng,https://www.polar polar.com/products/custom co...
1181950,eng,https://www.polar polar.com/products/custom co...
1181951,eng,https://www.polar polar.com/products/custom co...
1187087,zho,https://www.polarbear home.com/collections/cof...


# Evaluating Speedup on cleaning 100 million rows

In [16]:
print(f'Doing 100 million rows using 1 CPU would require time: {105.8 * 37 / 60:.1f} minutes')
print(f'Doing 100 million rows using Synthergent (5 processes): {105.8 * 11 / 60:.1f} minutes')
print(f'Doing 100 million rows using Synthergent (3 machines): {2.17:.1f} minutes')

Doing 100 million rows using 1 CPU would require time: 65.2 minutes
Doing 100 million rows using Synthergent (5 processes): 19.4 minutes
Doing 100 million rows using Synthergent (3 machines): 2.2 minutes
